# innov vs innov_newQC — OFA comparison (first 10 days)

Compares what GEOSldas wrote to ObsFcstAna under the original `hsaf_cdr_test_DAv8_M36_202006_innov`
run against the revised QC run `hsaf_cdr_test_DAv8_M36_202006_innov_newQC`.

The `innov_newQC` run currently only has June 2020 output (and partial July); we restrict the
comparison to **2020-06-01 through 2020-06-10**, the only days both runs share.

`innov_newQC` is the run we're treating as the new default going forward — this notebook is just
the sanity check on what changed before the deeper dive (a second notebook).


In [ ]:
import sys, os
from pathlib import Path


def _find_lib_root():
    cwd = Path(os.path.abspath(''))
    for p in [cwd] + list(cwd.parents):
        if (p / 'lib').exists() and (p / 'lib' / 'readers.py').exists():
            return p
        for child in p.glob('projects/*/lib'):
            if (child / 'readers.py').exists():
                return child.parent
    raise RuntimeError(f'Cannot find ascat_da/lib/ from {cwd}')


_root = _find_lib_root()
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

_repo_root = Path(_root).parents[1]
_common_io = _repo_root / 'common' / 'python' / 'io'
if str(_common_io) not in sys.path:
    sys.path.insert(0, str(_common_io))
from read_GEOSldas import read_tilecoord

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
# ── Configuration — edit here ─────────────────────────────────────────────────
START_DATE = '2020-06-01'
END_DATE   = '2020-06-10'

# Cache tags match the --version passed to build_ofa_cache.py
RUNS = {
    'innov':        {'version': 'v1',    'label': 'innov (original QC)'},
    'innov_newqc':  {'version': 'newqc', 'label': 'innov_newQC (revised QC)'},
}

CACHE_DIR = Path(_root) / '.cache' / 'ofa'
CACHE_TAG = f"{START_DATE.replace('-', '')}_{END_DATE.replace('-', '')}"

PRODUCTS = {
    'legacy': {'Metop-A': 9,  'Metop-B': 10, 'Metop-C': 11},
    'h121':   {'Metop-A': 14, 'Metop-B': 15, 'Metop-C': 16},
}
PLATFORM_COLOR = {'Metop-A': '#1f77b4', 'Metop-B': '#ff7f0e', 'Metop-C': '#2ca02c'}

# Tile lat/lon must come from the tilecoord file, not the OFA file's own lat/lon
# (the latter is the per-cycle super-ob center and jitters slightly within a tile).
# Both runs share the same M36 grid, so one tilecoord file covers both.
TILE_BASE = '/Users/amfox/Desktop/ASCAT_SSM_CDR/discover_sample/tilecoord/hsaf_cdr_test_DAv8_M36_202006_innov'
TILECOORD = f'{TILE_BASE}.ldas_tilecoord.bin'

print(f"Date range: {START_DATE} .. {END_DATE}")
for key, cfg in RUNS.items():
    print(f"  {key}: cache version={cfg['version']!r}")


## 1. Load cached OFA tile/cycle tables

Each run's OFA files were pre-aggregated to one row per (date, cycle, species, tile) with
`projects/ascat_da/scripts/build_ofa_cache.py`. If a cache is missing, rebuild it with:

```bash
python projects/ascat_da/scripts/build_ofa_cache.py \
  --start-date 2020-06-01 --end-date 2020-06-10 \
  --ofa-dir data/hsaf_cdr_test/<experiment_dir>/output/SMAP_EASEv2_M36_GLOBAL/ana/ens_avg/Y2020/M06 \
  --out-dir projects/ascat_da/.cache/ofa \
  --version <innov|newqc>
```


In [ ]:
def load_run_cache(version):
    pkl = CACHE_DIR / f'ofa_ascat_tile_cycle_{CACHE_TAG}_{version}.pkl'
    if not pkl.exists():
        raise FileNotFoundError(
            f'Missing OFA cache: {pkl}\n'
            'Run build_ofa_cache.py for this run/date range/version (see markdown above).'
        )
    return pd.read_pickle(pkl)


frames = []
for key, cfg in RUNS.items():
    df = load_run_cache(cfg['version'])
    df['run'] = key
    frames.append(df)
    print(f"{key}: {len(df):,} tile/cycle rows")

ofa = pd.concat(frames, ignore_index=True)


In [ ]:
tile_coord = read_tilecoord(TILECOORD)
tile_latlon = pd.DataFrame({
    'tilenum': tile_coord['tile_id'].astype('int64'),
    'tile_lat': tile_coord['com_lat'],
    'tile_lon': tile_coord['com_lon'],
}).drop_duplicates('tilenum')

ofa = ofa.merge(tile_latlon, on='tilenum', how='left')
n_missing = ofa['tile_lat'].isna().sum()
if n_missing:
    print(f"WARNING: {n_missing} OFA rows have no matching tilecoord entry")


## 2. Observation counts: did newQC drop obs?

Legacy BUFR ingestion is unchanged between runs, so legacy counts should match exactly. Any QC
change targeted H121 should show up as a count reduction for the H121 platforms only.


In [ ]:
counts = (
    ofa.groupby(['run', 'product', 'platform'], as_index=False)
    .agg(total_ofa_obs=('tilenum', 'size'))
)
pivot = counts.pivot_table(index=['product', 'platform'], columns='run', values='total_ofa_obs')
pivot['pct_change'] = 100 * (pivot['innov_newqc'] - pivot['innov']) / pivot['innov']
display(pivot.round(2))

fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(pivot))
width = 0.35
ax.bar(x - width / 2, pivot['innov'], width, label='innov')
ax.bar(x + width / 2, pivot['innov_newqc'], width, label='innov_newqc')
ax.set_xticks(x)
ax.set_xticklabels([f'{p}\n{plat}' for p, plat in pivot.index], fontsize=8)
ax.set_ylabel('Total OFA obs (10 days)')
ax.set_title('Obs counts by run')
ax.legend()
fig.tight_layout()


## 3. Maps: where did newQC remove H121 obs?

Per-tile obs counts (summed over all 10 days/cycles) for each H121 platform — `innov`, `innov_newqc`,
and the difference. Legacy is excluded here since section 2 already showed it's byte-for-byte
unchanged between runs.


In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.colors import LogNorm, TwoSlopeNorm

innov_df = ofa[ofa['run'] == 'innov']
newqc_df = ofa[ofa['run'] == 'innov_newqc']

h121_platforms = ['Metop-A', 'Metop-B', 'Metop-C']
h121_species = {plat: PRODUCTS['h121'][plat] for plat in h121_platforms}

def tile_counts(df, species_id):
    sub = df[df['species'] == species_id]
    return (
        sub.groupby('tilenum', as_index=False)
        .agg(lat=('tile_lat', 'first'), lon=('tile_lon', 'first'), n_obs=('tilenum', 'size'))
    )

tile_maps = {}
for plat, sp in h121_species.items():
    innov_t = tile_counts(innov_df, sp)
    newqc_t = tile_counts(newqc_df, sp)
    merged = innov_t.merge(newqc_t, on='tilenum', how='outer', suffixes=('_innov', '_newqc'))
    # recover lat/lon from whichever side has it (tiles missing entirely from one run)
    merged['lat'] = merged['lat_innov'].combine_first(merged['lat_newqc'])
    merged['lon'] = merged['lon_innov'].combine_first(merged['lon_newqc'])
    merged['n_obs_innov'] = merged['n_obs_innov'].fillna(0)
    merged['n_obs_newqc'] = merged['n_obs_newqc'].fillna(0)
    merged['diff'] = merged['n_obs_newqc'] - merged['n_obs_innov']
    tile_maps[plat] = merged

count_norm = LogNorm(vmin=1, vmax=max(m[['n_obs_innov', 'n_obs_newqc']].values.max() for m in tile_maps.values()))
diff_abs_max = max(m['diff'].abs().max() for m in tile_maps.values())
diff_norm = TwoSlopeNorm(vmin=-diff_abs_max, vcenter=0, vmax=diff_abs_max)

fig, axes = plt.subplots(
    3, 3, figsize=(15, 10),
    subplot_kw={'projection': ccrs.Robinson()},
)
row_specs = [
    ('n_obs_innov', 'innov', 'viridis', count_norm),
    ('n_obs_newqc', 'innov_newqc', 'viridis', count_norm),
    ('diff', 'newqc - innov', 'RdBu_r', diff_norm),
]

for row, (col, row_label, cmap, norm) in enumerate(row_specs):
    for ax_col, plat in enumerate(h121_platforms):
        ax = axes[row, ax_col]
        m = tile_maps[plat]
        plot_vals = m[col] if row < 2 else m[col]
        sc = ax.scatter(
            m['lon'], m['lat'], c=plot_vals, s=2, cmap=cmap, norm=norm,
            transform=ccrs.PlateCarree(),
        )
        ax.add_feature(cfeature.COASTLINE, linewidth=0.4)
        ax.set_extent([-180, 180, -60, 85], crs=ccrs.PlateCarree())
        if row == 0:
            ax.set_title(plat, fontsize=11)
        if ax_col == 0:
            ax.text(-0.08, 0.5, row_label, transform=ax.transAxes, rotation=90,
                     va='center', ha='center', fontsize=10)
    fig.colorbar(sc, ax=axes[row, :].tolist(), shrink=0.7, pad=0.01,
                 label='obs count' if row < 2 else 'Δ obs count')

fig.suptitle('H121 obs count per tile (10-day total): innov vs innov_newQC', y=0.99)


## 4. Tile-cycle match: what did newQC drop or add?

Outer-join on (date, cycle, species, tilenum) — the natural key for one OFA row — to see how
many tile-cycles are present in both runs vs. only one.


In [ ]:
KEYS = ['date', 'cycle', 'species', 'tilenum']

innov_df = ofa[ofa['run'] == 'innov']
newqc_df = ofa[ofa['run'] == 'innov_newqc']

matched = innov_df[KEYS + ['product', 'platform', 'obs_pct', 'innov_pct']].merge(
    newqc_df[KEYS + ['obs_pct', 'innov_pct']],
    on=KEYS, how='outer', suffixes=('_innov', '_newqc'), indicator=True,
)

match_summary = (
    matched.groupby(['product', 'platform', '_merge'], observed=True)
    .size()
    .unstack(fill_value=0)
    .rename(columns={'left_only': 'innov_only', 'right_only': 'newqc_only', 'both': 'both'})
)
display(match_summary)


## 5. Distribution shifts among matched tile-cycles

For tile-cycles present in *both* runs, compare obs and innov directly — this isolates
whether the surviving obs changed in character, separate from which obs survived QC.


In [ ]:
matched_both = matched[matched['_merge'] == 'both'].copy()
matched_both['obs_diff'] = matched_both['obs_pct_newqc'] - matched_both['obs_pct_innov']
matched_both['innov_diff'] = matched_both['innov_pct_newqc'] - matched_both['innov_pct_innov']

h121_platforms = ['Metop-A', 'Metop-B', 'Metop-C']
fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
for ax, plat in zip(axes, h121_platforms):
    sub = matched_both[(matched_both['product'] == 'h121') & (matched_both['platform'] == plat)]
    ax.hist(sub['obs_diff'].dropna(), bins=60, color=PLATFORM_COLOR[plat])
    ax.set_title(f'{plat}  (n={len(sub):,})')
    ax.set_xlabel('obs (newqc - innov)')
axes[0].set_ylabel('Tile-cycle count')
fig.suptitle('H121 obs shift on matched tile-cycles')
fig.tight_layout()


## 6. Coverage fraction: legacy vs H121 (innov vs innov_newQC)

Raw obs counts aren't comparable across products — H121 (12.5 km) and legacy BUFR (25 km) have
different swath/footprint densities even before QC. Instead, compute a resolution-agnostic
**coverage fraction** per tile: the share of the 10-day window's 80 analysis cycles
(10 days x 8 cycles) where *any* platform of that product produced a super-ob for that tile.
Legacy here combines Metop-A/B/C (species 9/10/11); H121 combines Metop-A/B/C (species 14/15/16).


In [ ]:
N_CYCLES_TOTAL = len(pd.date_range(START_DATE, END_DATE)) * 8


def coverage_fraction(df, species_list, n_total):
    sub = (
        df[df['species'].isin(species_list)][['tilenum', 'date', 'cycle', 'tile_lat', 'tile_lon']]
        .drop_duplicates(['tilenum', 'date', 'cycle'])
    )
    cov = sub.groupby('tilenum').agg(
        n_cycles=('date', 'size'), lat=('tile_lat', 'first'), lon=('tile_lon', 'first'),
    )
    cov['coverage_frac'] = cov['n_cycles'] / n_total
    return cov.reset_index()


legacy_species = list(PRODUCTS['legacy'].values())
h121_species = list(PRODUCTS['h121'].values())

cov_legacy = coverage_fraction(innov_df, legacy_species, N_CYCLES_TOTAL)
cov_innov_h121 = coverage_fraction(innov_df, h121_species, N_CYCLES_TOTAL)
cov_newqc_h121 = coverage_fraction(newqc_df, h121_species, N_CYCLES_TOTAL)

cov_panels = [
    ('Legacy (Metop-A/B/C)', cov_legacy),
    ('innov H121 (Metop-A/B/C)', cov_innov_h121),
    ('innov_newQC H121 (Metop-A/B/C)', cov_newqc_h121),
]

fig, axes = plt.subplots(3, 1, figsize=(22, 15), subplot_kw={'projection': ccrs.Robinson()})
for ax, (title, cov) in zip(axes, cov_panels):
    sc = ax.scatter(
        cov['lon'], cov['lat'], c=cov['coverage_frac'], s=0.5, cmap='viridis',
        vmin=0, vmax=1, transform=ccrs.PlateCarree(),
    )
    ax.add_feature(cfeature.COASTLINE, linewidth=0.4)
    ax.set_extent([-180, 180, -60, 85], crs=ccrs.PlateCarree())
    ax.set_title(f"{title}  (mean={cov['coverage_frac'].mean():.2f})", fontsize=11)
fig.colorbar(sc, ax=axes.tolist(), shrink=0.7, pad=0.02, label='fraction of cycles with a super-ob')
fig.suptitle('Per-tile coverage fraction (10-day window)', y=1.01)


## 7. Summary

In [ ]:
summary_rows = []
for (product, platform), grp in pivot.iterrows():
    summary_rows.append({
        'product': product,
        'platform': platform,
        'innov_obs': int(grp['innov']),
        'newqc_obs': int(grp['innov_newqc']),
        'pct_change': round(grp['pct_change'], 2),
    })
summary_df = pd.DataFrame(summary_rows)
display(summary_df)
print(
    "Legacy counts unchanged -> newQC only affects H121 ingestion/QC, as expected."
    if (summary_df.loc[summary_df['product'] == 'legacy', 'pct_change'].abs() < 1e-6).all()
    else "Unexpected: legacy counts changed too — check QC config diff."
)
